This note book for handling train test split, class imbalancing and numeric scaling

In [14]:
import pandas as pd

#import the dataset
cleaned_df = pd.read_csv('cleaned_dataset.csv')
cleaned_df.head()

,packet_length,src_ip,dst_ip,ip_version,ttl,ip_length,tos,id,flags_ip,fragment,...,psh,urg,udp_length,icmp_type,icmp_code,payload_size,Attack Type,is_tcp,is_udp,is_icmp
0,640,83,88,4.0,64.0,624.0,2.0,8876.0,2.0,0.0,...,0,0,-1.0,-1.0,-1.0,624,Benign,0,0,0
1,640,83,88,4.0,64.0,624.0,2.0,0.0,2.0,0.0,...,0,0,-1.0,-1.0,-1.0,624,Benign,0,0,0
2,64,83,88,4.0,64.0,48.0,2.0,0.0,2.0,0.0,...,0,0,-1.0,-1.0,-1.0,48,Benign,0,0,0
3,1856,86,88,4.0,64.0,1840.0,2.0,9204.0,2.0,0.0,...,0,0,-1.0,-1.0,-1.0,1840,Benign,0,0,0
4,64,83,90,4.0,64.0,48.0,2.0,0.0,2.0,0.0,...,0,0,-1.0,-1.0,-1.0,48,Benign,0,0,0


In [15]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Make sure the target column exists
if 'Attack Type' not in cleaned_df.columns:
    raise ValueError("The dataset must contain an 'Attack Type' column.")

# Separate features and target
X = cleaned_df.drop(columns=['Attack Type'])
y = cleaned_df['Attack Type']

# Stratified split by attack type
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Encode target labels to numeric values
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Class distribution in train:")
print(pd.Series(y_train_encoded).value_counts().sort_index())
print("\nEncoded classes:")
print(dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_))))

# Optional: save the split data
train_df = X_train.copy()
train_df['Attack Type'] = y_train
train_df['Attack Type_encoded'] = y_train_encoded

test_df = X_test.copy()
test_df['Attack Type'] = y_test
test_df['Attack Type_encoded'] = y_test_encoded

train_df.to_csv('train_set.csv', index=False)
test_df.to_csv('test_set.csv', index=False)

print("\nSaved train_set.csv and test_set.csv")


Train shape: (3713654, 29)
Test shape: (928414, 29)
Class distribution in train:
0     112000
1     627045
2    2648128
3     326481
Name: count, dtype: int64

Encoded classes:
{'Benign': np.int64(0), 'Bruteforce': np.int64(1), 'DDoS': np.int64(2), 'DOS': np.int64(3)}

Saved train_set.csv and test_set.csv


In [16]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

classes = np.unique(y_train_encoded)

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train_encoded
)

class_weights = dict(zip(classes, weights))

print(class_weights)

{np.int64(0): np.float64(8.28940625), np.int64(1): np.float64(1.480617021106938), np.int64(2): np.float64(0.3505923807308408), np.int64(3): np.float64(2.8436984081768926)}


In [17]:
from pathlib import Path
import joblib

save_dir = Path("saved")
save_dir.mkdir(exist_ok=True)
joblib.dump(label_encoder, save_dir / "label_encoder.pkl")

['saved\\label_encoder.pkl']

In [18]:
from sklearn.preprocessing import RobustScaler

# Scale continuous features using RobustScaler
continuous_cols = [
    "packet_length",
    "ip_version",
    "ttl",
    "ip_length",
    "tos",
    "id",
    "flags_ip",
    "fragment",
    "protocol",
    "src_port",
    "dst_port",
    "seq",
    "ack",
    "window",
    "udp_length",
    "icmp_type",
    "icmp_code",
    "payload_size"
]

save_dir = Path("saved")
save_dir.mkdir(parents=True, exist_ok=True)

output_dir = Path("data sets")
output_dir.mkdir(parents=True, exist_ok=True)

# Initialize the scaler
scaler = RobustScaler()

# Fit on training data and transform both sets
train_df[continuous_cols] = scaler.fit_transform(X_train[continuous_cols])
test_df[continuous_cols] = scaler.transform(X_test[continuous_cols])

# Save the scaler
joblib.dump(scaler, save_dir / "robust_scaler.pkl")

# Save scaled datasets
train_df.to_csv(output_dir / "scaled_train_set.csv", index=False)
test_df.to_csv(output_dir / "scaled_test_set.csv", index=False)

# Save the scaler
joblib.dump(scaler, save_dir / "robust_scaler.pkl")

# Save scaled datasets
train_df.to_csv(output_dir / "scaled_train_set.csv", index=False)
test_df.to_csv(output_dir / "scaled_test_set.csv", index=False)